## LexisNexis Fraud Detection - Task 2: Address Class Imbalance

Compares class weighting vs. SMOTE using a logistic regression baseline. Setup (data, features, preprocessing) is copied from Task 1. All preprocessing is fit on the 1994 training set only; SMOTE is applied to training data only.


In [1]:
!pip install -q imbalanced-learn

!rm -rf LexisNexis-1A-detecting-fraudulent-insurance-claims
!git clone -q https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git
%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [2]:
import json
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# M2 Task 1 saves both of these, so every notebook uses identical data and
# identical features. Do not copy feature code between notebooks, it already
# drifted once: this notebook was missing NewPolicy.
df = pd.read_csv("data/fraud_oracle_engineered.csv")

with open("data/model_features.json") as f:
    MODEL_FEATURES = json.load(f)

TARGET = "FraudFound_P"
print("Loaded", len(MODEL_FEATURES), "features from data/model_features.json")

train = df[df["Year"] == 1994].copy()
val   = df[df["Year"] == 1995].copy()
# 1996 is deliberately not loaded. It gets opened once, in Task 6.

X_train, y_train = train[MODEL_FEATURES], train[TARGET]
X_val,   y_val   = val[MODEL_FEATURES],   val[TARGET]

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

# only group columns that are actually in the feature list. PolicyType was
# dropped at the Challenge Advisor's request, so this guard stops the
# ColumnTransformer looking for a column that is not there.
grouped_categories = [c for c in ["Make", "PolicyType"]
                      if c in categorical_features]
regular_categories = [c for c in categorical_features
                      if c not in grouped_categories]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
grouped_category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                             min_frequency=0.01)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, regular_categories),
    ("grouped_categorical", grouped_category_pipeline, grouped_categories),
])

preprocessor.fit(X_train)          # training year only
X_train_processed = preprocessor.transform(X_train)
X_val_processed   = preprocessor.transform(X_val)

print("Grouped:", grouped_categories)
print("Train:", X_train_processed.shape, " Validation:", X_val_processed.shape)
print("Train fraud rate:", round(y_train.mean(), 4),
      "  fraud cases:", int(y_train.sum()))

Loaded 24 features from data/model_features.json
Grouped: ['Make']
Train: (6141, 88)  Validation: (5195, 88)
Train fraud rate: 0.0666   fraud cases: 409


Task 2: Class weighting vs. SMOTE

In [3]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

print("Train fraud rate:", round(y_train.mean(), 4))
print(y_train.value_counts())


Train fraud rate: 0.0666
FraudFound_P
0    5732
1     409
Name: count, dtype: int64


In [4]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score)

CAPACITY = 0.20   # how much of the claim book a review team can get through


def evaluate(name, model, X_tr, y_tr, X_ev, y_ev):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_ev)[:, 1]

    # the default 0.5 cutoff, which is NOT how this model would be used
    pred_half = (proba >= 0.5).astype(int)

    # a cutoff set by review capacity, which is how it would be used
    thresh = np.quantile(proba, 1 - CAPACITY)
    pred_cap = (proba >= thresh).astype(int)

    y_sorted = y_ev.to_numpy()[proba.argsort()[::-1]]
    cap10 = y_sorted[:int(len(y_sorted) * 0.10)].sum() / y_sorted.sum()
    cap20 = y_sorted[:int(len(y_sorted) * 0.20)].sum() / y_sorted.sum()

    return {
        "Approach": name,
        "PR-AUC": average_precision_score(y_ev, proba),
        "Caught top 10%": cap10,
        "Caught top 20%": cap20,
        "Precision @0.5": precision_score(y_ev, pred_half, zero_division=0),
        "Recall @0.5": recall_score(y_ev, pred_half),
        "F1 @0.5": f1_score(y_ev, pred_half),
        "Precision @cap": precision_score(y_ev, pred_cap, zero_division=0),
        "Recall @cap": recall_score(y_ev, pred_cap),
        "F1 @cap": f1_score(y_ev, pred_cap),
    }


lr_kwargs = dict(max_iter=1000, random_state=42)
results = []

results.append(evaluate("No handling", LogisticRegression(**lr_kwargs),
                        X_train_processed, y_train, X_val_processed, y_val))

results.append(evaluate("class_weight balanced",
                        LogisticRegression(class_weight="balanced", **lr_kwargs),
                        X_train_processed, y_train, X_val_processed, y_val))

X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_train_processed, y_train)
print("After SMOTE:", X_sm.shape, y_sm.value_counts().to_dict())
results.append(evaluate("SMOTE", LogisticRegression(**lr_kwargs),
                        X_sm, y_sm, X_val_processed, y_val))

results_df = pd.DataFrame(results).set_index("Approach").round(4)

After SMOTE: (11464, 88) {0: 5732, 1: 5732}


In [5]:
print("RANKING QUALITY. This is what matters, because the deliverable is a")
print("sorted list of claims, not a yes or no on each one.\n")
display(results_df[["PR-AUC", "Caught top 10%", "Caught top 20%"]])

print("\n\nAT THE DEFAULT 0.5 CUTOFF. Included to show why it misleads.")
print("Nothing scores above 0.5 on data this imbalanced, so the unweighted")
print("model flags almost nothing and recall looks like zero.\n")
display(results_df[["Precision @0.5", "Recall @0.5", "F1 @0.5"]])

print(f"\n\nAT A CUTOFF THAT FLAGS THE TOP {int(CAPACITY*100)}% OF CLAIMS.")
print("This is how the model would actually be used.\n")
display(results_df[["Precision @cap", "Recall @cap", "F1 @cap"]])

RANKING QUALITY. This is what matters, because the deliverable is a
sorted list of claims, not a yes or no on each one.



,PR-AUC,Caught top 10%,Caught top 20%
Approach,,,
No handling,0.1488,0.2824,0.5216
class_weight balanced,0.1393,0.2724,0.5083
SMOTE,0.1407,0.2924,0.4983




AT THE DEFAULT 0.5 CUTOFF. Included to show why it misleads.
Nothing scores above 0.5 on data this imbalanced, so the unweighted
model flags almost nothing and recall looks like zero.



,Precision @0.5,Recall @0.5,F1 @0.5
Approach,,,
No handling,0.2500,0.0033,0.0066
class_weight balanced,0.1265,0.7176,0.2150
SMOTE,0.1317,0.7176,0.2226




AT A CUTOFF THAT FLAGS THE TOP 20% OF CLAIMS.
This is how the model would actually be used.



,Precision @cap,Recall @cap,F1 @cap
Approach,,,
No handling,0.1511,0.5216,0.2343
class_weight balanced,0.1473,0.5083,0.2284
SMOTE,0.1444,0.4983,0.2239


Chosen strategy for Task 3 (PR-AUC first; recall breaks ties)

In [6]:
# Pick on ranking quality, not on the 0.5 numbers.
# The baseline stays IN the comparison on purpose. If neither method beats
# doing nothing, that is the finding, and we report it rather than picking
# the least bad of two options that both made things worse.
ranking = results_df[["PR-AUC", "Caught top 10%", "Caught top 20%"]]
best = ranking["PR-AUC"].idxmax()
gap = ranking.loc[best, "PR-AUC"] - ranking.loc["No handling", "PR-AUC"]

print("Best PR-AUC:", best, round(ranking.loc[best, "PR-AUC"], 4))
print("Gap against doing nothing:", round(gap, 4))

if gap <= 0.002:
    print("\nNeither method meaningfully improves ranking on logistic regression.")
    print("Recommendation: carry class_weight='balanced' into Tasks 3 and 4")
    print("because it is free and tree models respond to it differently,")
    print("but do not claim it improved the baseline.")

Best PR-AUC: No handling 0.1488
Gap against doing nothing: 0.0

Neither method meaningfully improves ranking on logistic regression.
Recommendation: carry class_weight='balanced' into Tasks 3 and 4
because it is free and tree models respond to it differently,
but do not claim it improved the baseline.


## Task 2: class weighting vs SMOTE

Fraud is about 6% of claims, so a model can score well while basically
ignoring it. We compared three approaches on logistic regression: no imbalance
handling, `class_weight="balanced"`, and SMOTE. All three trained on 1994 and
scored on 1995, using the 24 column feature list from Task 1. SMOTE was applied
to training data only. The 1996 test year was not touched.

### Result: neither method improved the model

| Approach | PR-AUC | Caught top 10% | Caught top 20% |
|---|---|---|---|
| No handling | **0.1488** | 0.2824 | **0.5216** |
| class_weight balanced | 0.1393 | 0.2724 | 0.5083 |
| SMOTE | 0.1407 | **0.2924** | 0.4983 |

Doing nothing has the best PR-AUC and the best top 20% capture. SMOTE edges
ahead on top 10% but loses on the other two, so there is no consistent winner
other than leaving it alone.

### Why they look like they helped

At the default 0.5 cutoff, class weighting takes recall from 0.0033 to 0.7176
and F1 from 0.0066 to 0.2150, which looks like an enormous improvement. It is
not. An unweighted model trained on 6% fraud produces scores that almost never
reach 0.5, so it flags almost nothing and recall is near zero by construction.
Weighting pushes every score upward so more of them cross the line. The model
did not get better at separating fraud from non-fraud, the cutoff moved
relative to the scores.

The proof is what happens when all three are given the same job, flag the top
20% of claims:

| Approach | Precision | Recall | F1 |
|---|---|---|---|
| No handling | **0.1511** | **0.5216** | **0.2343** |
| class_weight balanced | 0.1473 | 0.5083 | 0.2284 |
| SMOTE | 0.1444 | 0.4983 | 0.2239 |

They land within 0.011 of each other on F1 and the unweighted model is highest.

Worth stating plainly: the unweighted model at a 20% cutoff reaches F1 0.2343,
while the class weighted model at its flattering 0.5 cutoff only reaches
0.2150, and gets there while flagging roughly 33% of all claims instead of 20%.
Choosing a sensible cutoff on the plain model beats class weighting outright,
using a third less reviewer time.
